# Notebook 03 — Does Our Search Really Work?

**VidyaSankalp · Applied GenAI Engineering** · Tutor: Prudhvi Akella

In notebook 02 we asked one question and the results **looked** fine. Today we find out if they **are** fine — by
asking many questions whose right answers we already know, and counting.

```text
1 THE HOOK         notebook 02 hid a mistake
2 THE ANSWER KEY   questions + their right answers
3 ONE QUESTION     guess, search, pass or fail
4 ALL QUESTIONS    one number for each kind of question
5 THE REAL TEST ★  does the AI get everything it needs?
6 THE MISTAKES ★   why did questions fail? fix one
7 SCORECARD        green or red
8 SAVE THE RUN     MLflow
   Going further   (optional) more scores, languages, and more
```

**You need:** notebook 02 run once, your `.env` file, and `ground_truth_ecom.xlsx` in the `docs` folder.

In [ ]:
%pip install -q openai pinecone python-dotenv openpyxl mlflow

### ▶ Setup — just run this cell

This cell connects to OpenAI and Pinecone, opens the answer key, and prepares the search. **You don't need to read
it** — the chapters below explain everything you need. It takes about a minute.

In [ ]:
import os

import openpyxl
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone


# Read environment variables from the .env file in this folder.
# The file should contain OPENAI_API_KEY and PINECONE_API_KEY.
load_dotenv()

# Check that both API keys were loaded before trying to connect to either service.
# Raising an error here makes a missing key easier to diagnose.
for key in ("OPENAI_API_KEY", "PINECONE_API_KEY"):
    if not os.environ.get(key):
        raise ValueError(f"Please set {key} in your .env file.")


# Create the OpenAI client. It reads OPENAI_API_KEY from the environment.
client = OpenAI()

# These settings identify the embedding model and Pinecone index used by this notebook.
EMBED_MODEL = "text-embedding-3-small"  # Same model as notebook 02
INDEX_NAME = "ecom-kb-tutorial"
NAMESPACE = "ecom-kb"

# Connect to Pinecone, then select the index this notebook will search.
pinecone = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pinecone.Index(INDEX_NAME)


# Open the Excel workbook that contains the expected answers and test questions.
# read_only=True uses less memory because this code only needs to read the file.
answer_key = openpyxl.load_workbook(
    "./docs/ground_truth_ecom.xlsx",
    read_only=True,
)


def read_sheet(sheet_name):
    """
    Read one worksheet from the answer-key workbook.

    Input:
        sheet_name: The name of the worksheet to read, such as "golden".

    Process:
        Treat the first row as column headings. Turn each later, non-empty row
        into a dictionary that maps each heading to its cell value.

    Returns:
        A list of dictionaries. For example:
        [{"question": "What is ...?", "answer": "..."}]
    """
    # Get the worksheet by name and read its rows as tuples of cell values.
    # values_only=True returns the cell contents, rather than Excel cell objects.
    rows = answer_key[sheet_name].iter_rows(values_only=True)

    # The first row contains headings such as "question" and "answer".
    # Save them so they can become keys in each row's dictionary.
    column_names = next(rows)

    records = []

    # Go through each remaining row in the worksheet.
    for values in rows:
        # Check whether this row contains anything.
        # Completely blank rows should not become empty dictionaries.
        row_has_data = False
        for value in values:
            if value is not None:
                row_has_data = True
                break

        # Keep only rows that have at least one value.
        if row_has_data:
            # Pair each heading with the value in the same position.
            # For example, ("question", "answer") and ("How?", "This way.")
            # become {"question": "How?", "answer": "This way."}.
            record = dict(zip(column_names, values))
            records.append(record)

    # Return all non-empty worksheet rows as a list of dictionaries.
    return records


def split_ids(cell):
    """
    Turn a worksheet cell containing IDs into a Python list of IDs.

    Input:
        cell: A cell value that may contain one ID per line.

    Process:
        Convert the value to text, split it at newline characters, trim spaces,
        and discard blank lines.

    Returns:
        A list of cleaned IDs. An empty or missing cell returns [].
    """
    # Empty cells have no IDs to split.
    if not cell:
        return []

    ids = []

    # Convert to text in case Excel stored the cell as a number or another type.
    # Then split the text wherever there is a newline.
    for part in str(cell).split("\n"):
        # Remove spaces before and after each ID.
        cleaned_part = part.strip()

        # Ignore blank lines, but keep lines that contain an ID.
        if cleaned_part:
            ids.append(cleaned_part)

    return ids


# Load each worksheet into a list of dictionaries.
# Each variable contains the rows from one named worksheet.
glossary_questions = read_sheet("glossary_questions")
example_questions = read_sheet("example_questions")
join_questions = read_sheet("join_questions")
schema_questions = read_sheet("schema_questions")
golden_questions = read_sheet("golden")
language_questions = read_sheet("languages")


# Collect all IDs stored in the Pinecone namespace.
# A set is used because IDs should be unique, and later we only need to check
# whether a particular ID exists.
ALL_IDS = set()

# Pinecone returns IDs in batches, so this loop processes each batch.
for id_batch in index.list(namespace=NAMESPACE):
    for record_id in id_batch:
        ALL_IDS.add(record_id)

# An empty namespace usually means notebook 02 has not populated the index yet.
# Stop here so later searches do not run against missing data.
if not ALL_IDS:
    raise ValueError(
        f"Namespace '{NAMESPACE}' is empty — run notebook 02 first."
    )


# Gather all questions from the six worksheets into one list.
# The same question can appear on multiple sheets, so only add it once.
all_questions = []

question_sheets = [
    glossary_questions,
    example_questions,
    join_questions,
    schema_questions,
    golden_questions,
    language_questions,
]

for sheet in question_sheets:
    for row in sheet:
        question = row["question"]

        # Avoid making duplicate embedding requests for repeated questions.
        if question not in all_questions:
            all_questions.append(question)


# Create and store an embedding vector for every unique test question.
# Pinecone uses these vectors to find records with similar meaning.
VECTOR = {}

# Send questions in groups of 100. Batching reduces the number of API calls.
for start in range(0, len(all_questions), 100):
    # Take up to 100 questions, beginning at the current position.
    batch = all_questions[start : start + 100]

    # Ask OpenAI to convert each question into a numeric embedding vector.
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=batch,
    )

    # The response has one vector for each question in the batch.
    # Store each vector under its question so search() can look it up later.
    for question, item in zip(batch, response.data):
        VECTOR[question] = item.embedding


# These lists tell search() which kinds of records to search.
# The values must match the "sheet" metadata stored in Pinecone.
SCHEMA_TABS = ["nlq_table", "nlq_column", "nlq_join"]
EXAMPLE_TABS = ["nlq_examples"]
GLOSSARY_TABS = ["glossary", "capability_card"]


def search(question, tabs, top_k=10):
    """
    Search Pinecone for records related to a question.

    Inputs:
        question: A question whose embedding was created earlier.
        tabs: The sheet names to include in this search.
        top_k: The maximum number of matching records to return.

    Process:
        Look up the question's saved vector, then ask Pinecone for similar
        records whose "sheet" metadata matches one of the requested tabs.

    Returns:
        Pinecone's matches, ordered from the best match to lower-ranked matches.
    """
    results = index.query(
        # Use the vector created earlier for this exact question.
        vector=VECTOR[question],

        # Ask Pinecone to return up to this many matches.
        top_k=top_k,

        # Search only inside the configured namespace.
        namespace=NAMESPACE,

        # Include metadata such as the sheet name and glossary mappings.
        include_metadata=True,

        # Restrict results to records whose sheet is in the tabs list.
        filter={"sheet": {"$in": tabs}},
    )

    # Return the matching records. Each match includes its ID, score, and metadata.
    return results.matches


def ids_of(matches):
    """
    Extract Pinecone record IDs from a list of search matches.

    Input:
        matches: Search results returned by search().

    Returns:
        A list containing each match's ID, in the same order as the matches.
        Since search results are ranked, the first ID is the best match.
    """
    record_ids = []

    # Each match is an object with an "id" field.
    for match in matches:
        record_ids.append(match.id)

    return record_ids


# These capability-card records are added to every context pack.
# They provide general information that should always be available.
ALWAYS_ADDED = [
    "capability_card::1",
    "capability_card::2",
    "capability_card::3",
]

# Limit the number of example records added to each context pack.
EXAMPLES_IN_PACK = 2


def context_pack(question):
    """
    Search for useful records and assemble their IDs into a context pack.

    Input:
        question: The test question to search for.

    Process:
        1. Search schema, example, and glossary records separately.
        2. Select the highest-ranked IDs from each search.
        3. Follow glossary links to related column records when those records exist.
        4. Return the selected IDs, full ranked ID lists, and best search score.

    Returns:
        A dictionary containing the IDs and scores used to build the context.
    """
    # Search each category separately. This lets each category have its own
    # result limit when records are added to the context pack.
    schema_matches = search(question, SCHEMA_TABS)
    example_matches = search(question, EXAMPLE_TABS)
    glossary_matches = search(question, GLOSSARY_TABS)

    # Start with an empty list, then add the top results from each category.
    found_by_search = []

    # Include up to three of the best schema records.
    schema_ids = ids_of(schema_matches)
    for record_id in schema_ids[:3]:
        found_by_search.append(record_id)

    # Include up to EXAMPLES_IN_PACK of the best example records.
    example_ids = ids_of(example_matches)
    for record_id in example_ids[:EXAMPLES_IN_PACK]:
        found_by_search.append(record_id)

    # Include up to three of the best glossary or capability-card records.
    glossary_ids = ids_of(glossary_matches)
    for record_id in glossary_ids[:3]:
        found_by_search.append(record_id)

    # A glossary record can name related columns in its "maps_to" metadata.
    # Follow those references so the context also includes the linked column
    # records, but only when the referenced record exists in Pinecone.
    followed_links = []

    # Only inspect the three highest-ranked glossary search results.
    for match in glossary_matches[:3]:

        # Only glossary records have the "maps_to" links we want to follow.
        if match.metadata.get("sheet") == "glossary":

            # get() returns an empty list if the record has no "maps_to" field.
            mapped_columns = match.metadata.get("maps_to", [])

            for column in mapped_columns:
                # Build the Pinecone ID for the linked column record.
                column_id = "column::" + column

                # Do not add links to records that are missing from the index.
                if column_id in ALL_IDS:
                    followed_links.append(column_id)

    # Find the highest score among the first result from each search.
    # Each search result list is expected to contain at least one match.
    best_score = max(
        schema_matches[0].score,
        example_matches[0].score,
        glossary_matches[0].score,
    )

    # Return both the selected context IDs and the full ranked search results.
    return {
        # IDs selected from the three searches for the context pack.
        "found_by_search": found_by_search,

        # Column IDs reached by following glossary "maps_to" links.
        "followed_links": followed_links,

        # Capability cards that should always be added.
        "always_added": ALWAYS_ADDED,

        # Full ranked ID lists, useful for inspecting search quality.
        "schema_top": schema_ids,
        "example_top": example_ids,
        "glossary_top": glossary_ids,

        # The highest top-result score from the three searches.
        "best_score": best_score,
    }


# Confirm that the namespace and question embeddings are ready.
print(
    f"✅ Ready: {len(ALL_IDS)} records in Pinecone, "
    f"{len(VECTOR)} test questions prepared"
)

In [ ]:
def read_sheet(sheet_name):
    """
    Read one worksheet from the answer-key workbook.

    Input:
        sheet_name: The name of the worksheet to read, such as "golden".

    Process:
        Treat the first row as column headings. Turn each later, non-empty row
        into a dictionary that maps each heading to its cell value.

    Returns:
        A list of dictionaries. For example:
        [{"question": "What is ...?", "answer": "..."}]
    """
    # Get the worksheet by name and read its rows as tuples of cell values.
    # values_only=True returns the cell contents, rather than Excel cell objects.
    rows = answer_key[sheet_name].iter_rows(values_only=True)

    # The first row contains headings such as "question" and "answer".
    # Save them so they can become keys in each row's dictionary.
    column_names = next(rows)

    records = []

    # Go through each remaining row in the worksheet.
    for values in rows:
        # Check whether this row contains anything.
        # Completely blank rows should not become empty dictionaries.
        row_has_data = False
        for value in values:
            if value is not None:
                row_has_data = True
                break

        # Keep only rows that have at least one value.
        if row_has_data:
            # Pair each heading with the value in the same position.
            # For example, ("question", "answer") and ("How?", "This way.")
            # become {"question": "How?", "answer": "This way."}.
            record = dict(zip(column_names, values))
            records.append(record)

    # Return all non-empty worksheet rows as a list of dictionaries.
    return records

In [ ]:
def split_ids(cell):
    """
    Turn a worksheet cell containing IDs into a Python list of IDs.

    Input:
        cell: A cell value that may contain one ID per line.

    Process:
        Convert the value to text, split it at newline characters, trim spaces,
        and discard blank lines.

    Returns:
        A list of cleaned IDs. An empty or missing cell returns [].
    """
    # Empty cells have no IDs to split.
    if not cell:
        return []

    ids = []

    # Convert to text in case Excel stored the cell as a number or another type.
    # Then split the text wherever there is a newline.
    for part in str(cell).split("\n"):
        # Remove spaces before and after each ID.
        cleaned_part = part.strip()

        # Ignore blank lines, but keep lines that contain an ID.
        if cleaned_part:
            ids.append(cleaned_part)

    return ids

In [ ]:
# ============================================================================================
# SETUP — run this cell once. You don't need to read it.
# ============================================================================================
import csv
import math
import os
import re
from datetime import datetime

import openpyxl
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone

load_dotenv()
for key in ("OPENAI_API_KEY", "PINECONE_API_KEY"):
    if not os.environ.get(key):
        raise ValueError(f"Please set {key} in your .env file.")

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"      # the same model as notebook 02
INDEX_NAME = "ecom-kb-tutorial"
NAMESPACE = "ecom-kb"
index = Pinecone(api_key=os.getenv("PINECONE_API_KEY")).Index(INDEX_NAME)

# --- the answer key -------------------------------------------------------------------------
answer_key = openpyxl.load_workbook("./docs/ground_truth_ecom.xlsx", read_only=True)


def read_sheet(sheet_name):
    """The rows of one sheet, as a list of dictionaries."""
    rows = answer_key[sheet_name].iter_rows(values_only=True)
    column_names = next(rows)
    records = []
    for values in rows:
        if any(value is not None for value in values):
            records.append(dict(zip(column_names, values)))
    return records


def split_ids(cell):
    """A cell can hold several ids, one per line -> a Python list."""
    if not cell:
        return []
    return [part.strip() for part in str(cell).split("\n") if part.strip()]


glossary_questions = read_sheet("glossary_questions")
example_questions = read_sheet("example_questions")
join_questions = read_sheet("join_questions")
schema_questions = read_sheet("schema_questions")
golden_questions = read_sheet("golden")
language_questions = read_sheet("languages")

# --- every record id in Pinecone ------------------------------------------------------------
ALL_IDS = set()
for id_batch in index.list(namespace=NAMESPACE):
    ALL_IDS.update(id_batch)
if not ALL_IDS:
    raise ValueError(f"Namespace '{NAMESPACE}' is empty — run notebook 02 first.")

# --- embed every test question once (100 per call) ------------------------------------------
all_questions = []
for sheet in (glossary_questions, example_questions, join_questions, schema_questions,
              golden_questions, language_questions):
    for row in sheet:
        if row["question"] not in all_questions:
            all_questions.append(row["question"])

VECTOR = {}
for start in range(0, len(all_questions), 100):
    batch = all_questions[start : start + 100]
    response = client.embeddings.create(model=EMBED_MODEL, input=batch)
    for question, item in zip(batch, response.data):
        VECTOR[question] = item.embedding

# --- the agent's three searches (same tabs as notebook 02) ---------------------------------
SCHEMA_TABS = ["nlq_table", "nlq_column", "nlq_join"]
EXAMPLE_TABS = ["nlq_examples"]
GLOSSARY_TABS = ["glossary", "capability_card"]


def search(question, tabs, top_k=10):
    """Search Pinecone in the given tabs. Returns the list of matching ids, best first."""
    results = index.query(vector=VECTOR[question], top_k=top_k, namespace=NAMESPACE,
                          include_metadata=True, filter={"sheet": {"$in": tabs}})
    return results.matches


def ids_of(matches):
    return [match.id for match in matches]


# --- the context pack, as ids (the same three routes as notebook 02, chapter 9) -------------
ALWAYS_ADDED = ["capability_card::1", "capability_card::2", "capability_card::3"]
EXAMPLES_IN_PACK = 2


def context_pack(question):
    schema = search(question, SCHEMA_TABS)
    examples = search(question, EXAMPLE_TABS)
    glossary = search(question, GLOSSARY_TABS)

    found_by_search = ids_of(schema)[:3] + ids_of(examples)[:EXAMPLES_IN_PACK] + ids_of(glossary)[:3]

    followed_links = []
    for match in glossary[:3]:
        if match.metadata.get("sheet") == "glossary":
            for column in match.metadata.get("maps_to", []):
                if "column::" + column in ALL_IDS:
                    followed_links.append("column::" + column)

    return {"found_by_search": found_by_search, "followed_links": followed_links,
            "always_added": ALWAYS_ADDED, "schema_top": ids_of(schema), "example_top": ids_of(examples),
            "glossary_top": ids_of(glossary),
            "best_score": max(schema[0].score, examples[0].score, glossary[0].score)}


print(f"✅ Ready: {len(ALL_IDS)} records in Pinecone, {len(VECTOR)} test questions prepared")

## 1 — The hook: notebook 02 hid a mistake

In notebook 02 we asked **"What was our GMV last month?"** The answer needs the column `orders.total_amount` — that
is where revenue is stored.

**Guess first:** when we search the **tables and columns** for this question, will `orders.total_amount` be in the
top 3? Then run the cell.

In [ ]:
question = "What was our GMV last month?"
top_3 = ids_of(search(question, SCHEMA_TABS))[:3]

print("Question:", question)
print("Top 3 tables/columns found:")
for record_id in top_3:
    print("   ", record_id)

print()
if "column::ecom.orders.total_amount" in top_3:
    print("✅ orders.total_amount was found")
else:
    print("❌ orders.total_amount was NOT found — the search missed the most important column")

In the real run, the search **missed** it. Notebook 02 still gave the AI the right column — through the glossary's
link (Revenue → `orders.total_amount`). But we only know that because we **looked closely at one question**.

Users will ask thousands of different questions. We can't look at each one by hand. So we need a list of questions
**with the right answers written down**, and a way to **count**.

## 2 — The answer key

`docs/ground_truth_ecom.xlsx` is our answer key: test questions, and for each one the **id** of the record that
should be found. Open it in Excel if you like. Here are three rows:

In [ ]:
for row in glossary_questions[:3]:
    print("Question:    ", row["question"])
    print("Right answer:", row["right_id"])
    print()

Notice: the first question asks about **"merchandise, before delivery charges and tax"** — it never says
"Revenue". That is on purpose.

**The one rule:** a test question must never be the same text we stored. Otherwise the search finds it by simply
copying — like an exam where the questions are printed in the textbook. Our questions use **other words with the
same meaning**, the way real users talk.

The answer key has questions of different kinds:

| Sheet | How many | Example |
|---|---:|---|
| `glossary_questions` | 66 | "How much does a typical checkout bring in?" → Average Order Value |
| `example_questions` | 61 | "Which ten products sold the most pieces?" → a saved SQL example |
| `join_questions` | 16 | "Which warehouse city sends the most parcels?" → a join between two tables |
| `golden` | 49 | real business questions, like a manager would ask |

## 3 — One question: guess, search, pass or fail

The AI only ever sees the **top 3** search results. So our rule is simple:

```text
right answer in the top 3?   ✅ PASS
not in the top 3?            ❌ FAIL
```

**Guess first:** *"How much does a typical checkout bring in?"* — the right answer is `glossary::Average Order Value`.
The question doesn't use any of those words. Will the search still find it?

In [ ]:
row = glossary_questions[2]          # change the number (0 to 65) to try other questions
question = row["question"]
right_answer = row["right_id"]

top_3 = ids_of(search(question, GLOSSARY_TABS))[:3]

print("Question:    ", question)
print("Right answer:", right_answer)
print()
for place, record_id in enumerate(top_3, start=1):
    mark = "  ← right answer" if record_id == right_answer else ""
    print(f"  {place}. {record_id}{mark}")
print()
print("✅ PASS" if right_answer in top_3 else "❌ FAIL")

**Try it:** change `glossary_questions[2]` to another number between 0 and 65 and run the cell again. Guess before
each run. Some will pass, some will fail.

Doing this 66 times by hand would be boring. Let the computer do it.

## 4 — All questions: one number for each kind

Same check — "is the right answer in the top 3?" — for every question. Then we count:

```text
passed 50 of 66   →   50 ÷ 66 = 0.76   (1.00 would mean every question passed)
```

In [ ]:
def check_all(sheet_name, questions, tabs):
    """Check every question of one sheet. Returns one result per question."""
    results = []
    for row in questions:
        top_10 = ids_of(search(row["question"], tabs))
        right_answers = split_ids(row["right_id"])

        passed = 0
        for record_id in top_10[:3]:
            if record_id in right_answers:
                passed = 1

        results.append({"set": sheet_name, "qid": row["qid"], "question": row["question"],
                        "right_answer": " | ".join(right_answers), "top_3": " | ".join(top_10[:3]),
                        "top_10": top_10, "right_answers": right_answers, "passed": passed})
    return results


def count_passed(results):
    passed = 0
    for result in results:
        if result["passed"] == 1:
            passed += 1
    return passed


glossary_results = check_all("glossary", glossary_questions, GLOSSARY_TABS)
example_results = check_all("examples", example_questions, EXAMPLE_TABS)
join_results = check_all("joins", join_questions, SCHEMA_TABS)

print("In the top 3:")
for name, results in (("glossary", glossary_results), ("examples", example_results), ("joins", join_results)):
    passed = count_passed(results)
    print(f"  {name:9} passed {passed:3} of {len(results):3}  =  {passed / len(results):.2f}")

**What to notice:**

- **examples** usually scores high: example questions are written the way users talk.
- **glossary** is harder: our questions never use the business word, so the search must understand the **meaning**.
- **joins** is usually lowest: every join text starts with "Join A to B on …", so they all look alike.

> **But:** the AI doesn't get just one record. It gets a whole **context pack**. Does the pack have everything?

## 5 — The real test: does the AI get everything it needs? ★

Each `golden` question lists the records the AI **must** have. The question passes only if **all** of them are in
the context pack.

```text
"What was our GMV last month?"
   must have:   glossary::Revenue                    what GMV means here
                column::ecom.orders.total_amount     where the number is stored

context pack = found by search  +  followed links  +  always added   (notebook 02, chapter 9)

both in the pack?   ✅ PASS      2 of 2 = 1.0
only one?           ❌ FAIL      1 of 2 = 0.5   ← this number is the "pack completeness"
```

**Guess first:** for the GMV question — which route brings in `orders.total_amount`?

In [ ]:
question = "What was our GMV last month?"
must_have = split_ids(golden_questions[0]["must_find"])
pack = context_pack(question)

for record_id in must_have:
    if record_id in pack["found_by_search"]:
        how = "✅ found by search"
    elif record_id in pack["followed_links"]:
        how = "✅ followed link"
    elif record_id in pack["always_added"]:
        how = "✅ always added"
    else:
        how = "❌ MISSING"
    print(f"{record_id:40} {how}")

Now all the golden questions the AI should answer. We count twice — **without** the links and **with** them — to see
what the links from notebook 02 are worth.

In [ ]:
def share_in(needed, pack_ids):
    """What share of the needed ids is in the pack: 2 of 2 -> 1.0, 1 of 2 -> 0.5."""
    found = 0
    for record_id in needed:
        if record_id in pack_ids:
            found += 1
    return found / len(needed)


golden_results = []
for row in golden_questions:
    must_have = split_ids(row["must_find"])
    if row["behaviour"] == "refuse" or len(must_have) == 0:
        continue                                    # questions to refuse are in "Going further"

    pack = context_pack(row["question"])
    search_only = pack["found_by_search"]
    with_links = search_only + pack["followed_links"] + pack["always_added"]

    missing = []
    for record_id in must_have:
        if record_id not in with_links:
            missing.append(record_id)

    golden_results.append({"set": "golden", "qid": row["qid"], "question": row["question"], "type": row["type"],
                           "search_only": share_in(must_have, search_only),
                           "completeness": share_in(must_have, with_links),
                           "missing": " | ".join(missing), "top_3": " | ".join(pack["glossary_top"][:3]),
                           "passed": 1 if len(missing) == 0 else 0})


def average(results, key):
    total = 0
    for result in results:
        total += result[key]
    return total / len(results)


print(f"{len(golden_results)} business questions")
print(f"  pack completeness — search only:   {average(golden_results, 'search_only'):.2f}")
print(f"  pack completeness — with links:    {average(golden_results, 'completeness'):.2f}")
print(f"  questions with EVERYTHING needed:  {count_passed(golden_results)} of {len(golden_results)}")

**What to notice:** the jump from "search only" to "with links" is what the `maps_to` links from notebook 02 are
worth — records the search missed, but the links still brought in. That design choice is now **measured**, not
just argued.

> **Next:** which questions failed, and why?

## 6 — The mistakes ★

A score tells you **how much** is wrong. The mistakes tell you **what**. Here are five failed business questions.
For each one, **guess why** before reading the table below.

In [ ]:
shown = 0
for result in golden_results:
    if result["passed"] == 0 and shown < 5:
        print(f"{result['qid']}  {result['question']}")
        print(f"     missing:               {result['missing']}")
        print(f"     glossary search top 3: {result['top_3']}")
        print()
        shown += 1

# Save every mistake to a file you can open in Excel.
os.makedirs("eval_runs", exist_ok=True)
mistakes = []
for results in (golden_results, glossary_results, example_results, join_results):
    for result in results:
        if result["passed"] == 0:
            mistakes.append(result)
with open("eval_runs/misses.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["set", "qid", "question", "right_answer", "missing", "top_3"],
                            extrasaction="ignore")
    writer.writeheader()
    writer.writerows(mistakes)
print(f"All {len(mistakes)} mistakes saved to eval_runs/misses.csv")

Almost every mistake is one of three kinds:

| What you see | Why | Fix |
|---|---|---|
| the user's word (like "OTD", "BOPIS") isn't in the record | the stored text doesn't know that word | add it to the term's **synonyms** in `kb_feed_ecom.xlsx` |
| the glossary term was found, but its column is missing | the link is missing | add the column to the term's **maps_to** |
| a join is missing (question about several tables) | similar is not the same as connected | the agent follows joins in code (Agent V3) |

**The good news:** most fixes are in the **Excel file**, not in the code.

**Try it:** pick one mistake. Fix it in `kb_feed_ecom.xlsx`. Run notebook 02 again (to reload), then this notebook.
Did it pass?

## 7 — Scorecard

We decide what is "good enough" **before** looking at the results — otherwise we'd just pick a number below
whatever we got.

In [ ]:
TARGETS = {
    "business questions: pack completeness": (average(golden_results, "completeness"), 0.90),
    "glossary: in top 3": (count_passed(glossary_results) / len(glossary_results), 0.80),
    "examples: in top 3": (count_passed(example_results) / len(example_results), 0.80),
    "joins: in top 3": (count_passed(join_results) / len(join_results), 0.60),
}

targets_met = 0
for name, (score, target) in TARGETS.items():
    if score >= target:
        targets_met += 1
        print(f"✅ {name:40} {score:.2f}   (target {target:.2f})")
    else:
        print(f"❌ {name:40} {score:.2f}   (target {target:.2f})")
print()
print(f"{targets_met} of {len(TARGETS)} targets met")

A ❌ is not a failure — it is your **to-do list**. Open `eval_runs/misses.csv`, fix the knowledge, run again.

## 8 — Save the run

Each time you run this notebook you get new scores. **MLflow** saves every run, so you can compare them side by side
in your browser — for example before and after a fix. It runs on your computer; no account needed.
See `README_MLflow.md` for how to open it.

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")            # a file next to this notebook
mlflow.set_experiment("vectordb-03-retrieval-eval")
run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")

with mlflow.start_run(run_name=run_name):
    # the settings used
    mlflow.log_param("embed_model", EMBED_MODEL)
    mlflow.log_param("examples_in_pack", EXAMPLES_IN_PACK)

    # the scores
    mlflow.log_metric("golden_pack_completeness", TARGETS["business questions: pack completeness"][0])
    mlflow.log_metric("golden_search_only", average(golden_results, "search_only"))
    mlflow.log_metric("glossary_in_top_3", TARGETS["glossary: in top 3"][0])
    mlflow.log_metric("examples_in_top_3", TARGETS["examples: in top 3"][0])
    mlflow.log_metric("joins_in_top_3", TARGETS["joins: in top 3"][0])
    mlflow.log_metric("targets_met", targets_met)

    # every question: passed or not
    table = {"set": [], "qid": [], "question": [], "passed": []}
    for results in (golden_results, glossary_results, example_results, join_results):
        for result in results:
            for column in table:
                table[column].append(result[column])
    mlflow.log_table(data=table, artifact_file="per_question.json")

    # the mistakes file
    mlflow.log_artifact("eval_runs/misses.csv")

print("✅ Saved to MLflow as", run_name)
print("To look at it: mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001   (see README_MLflow.md)")

## What you learned

Looking at one question isn't proof → we wrote down the **right answers first** → we checked hundreds of questions
and **counted** → the AI got most of what it needed, and the **links** added a measurable part → the **mistakes**
showed what to fix, mostly in the Excel file → every run is **saved** to compare.

## Quick check

1. Why isn't one good-looking question enough?
2. Why must a test question never be the exact text we stored?
3. What does "in top 3 = 0.76" mean?
4. What is pack completeness?
5. Most mistakes are fixed where — in the code or in the Excel file?

<details><summary><b>Answers</b> (click)</summary>

1. It is only one example — the GMV search missed a column, and only many questions show how often that happens.
2. The search would find it by copying, not by understanding.
3. For 76% of the questions, the right answer was in the top 3 results.
4. Of the records the AI must have, the share that is in the context pack (2 of 2 = 1.0).
5. In the Excel file: a missing synonym or a missing `maps_to` link.

</details>

**Next:** Agent Version 3 uses these searches as tools — and checks the last part: the SQL and the answer.

---

# Going further (optional)

The main part is done. Each section below is short and stands alone — pick what interests you.

### A — Is 1st place better than 3rd? (rank score and nDCG)

"In top 3" treats 1st and 3rd place the same. Two scores that don't:

```text
rank score (MRR)   1 ÷ the place:   1st → 1.0    2nd → 0.5    3rd → 0.33    not found → 0
found first        1 if the right answer is 1st, else 0
```

In [ ]:
def rank_score(top_ids, right_answers):
    place = 1
    for record_id in top_ids:
        if record_id in right_answers:
            return 1 / place
        place += 1
    return 0


for name, results in (("glossary", glossary_results), ("examples", example_results), ("joins", join_results)):
    total_rank = 0
    total_first = 0
    for result in results:
        total_rank += rank_score(result["top_10"], result["right_answers"])
        if result["top_10"] and result["top_10"][0] in result["right_answers"]:
            total_first += 1
    print(f"{name:9} rank score {total_rank / len(results):.2f}   found first {total_first / len(results):.2f}")

**nDCG** goes one step further: some records matter more (must-have = 2 points, helpful = 1 point), and points count
less the lower they are in the list: `points ÷ log2(place + 1)`. nDCG = our points ÷ the best possible points. It is
1.0 when the most important records are at the top.

### B — Is the embedding model better than counting words?

The simplest search: count the words the question and the record share. If our embedding search doesn't beat this,
the model isn't worth its cost.

In [ ]:
SMALL_WORDS = {"a", "an", "the", "of", "to", "in", "on", "for", "by", "with", "and", "or", "is", "are", "was", "do",
               "does", "did", "we", "our", "us", "how", "what", "which", "who", "when", "where", "much", "many",
               "each", "per", "from", "at", "it", "this", "that", "have", "has", "more", "most", "not"}

# The stored text of every record.
stored = {}
id_list = sorted(ALL_IDS)
for start in range(0, len(id_list), 100):
    for record_id, record in index.fetch(ids=id_list[start : start + 100], namespace=NAMESPACE).vectors.items():
        stored[record_id] = record.metadata


def words_of(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower())) - SMALL_WORDS


def word_search_top_3(question, tabs):
    scored = []
    for record_id, metadata in stored.items():
        if metadata.get("sheet") in tabs:
            shared = len(words_of(question) & words_of(metadata.get("text", "")))
            scored.append((-shared, record_id))
    scored.sort()
    return [record_id for minus_shared, record_id in scored[:3]]


print("In top 3:     word search   embeddings")
for name, questions, results, tabs in (("glossary", glossary_questions, glossary_results, GLOSSARY_TABS),
                                       ("examples", example_questions, example_results, EXAMPLE_TABS)):
    word_passed = 0
    for row in questions:
        if row["right_id"] in word_search_top_3(row["question"], tabs):
            word_passed += 1
    print(f"  {name:10} {word_passed / len(questions):10.2f} {count_passed(results) / len(results):12.2f}")

### C — Questions the AI should refuse

Some golden questions can't be answered from our data ("How many people visited our website yesterday?") or have
nothing to do with it ("Who won the cricket match?"). Does the search find the "WHAT IT CANNOT ANSWER" card?

In [ ]:
for row in golden_questions:
    if row["behaviour"] == "refuse":
        pack = context_pack(row["question"])
        card_found = "capability_card::2" in pack["glossary_top"][:3]
        mark = "✅ limits card found" if card_found else "—  limits card not in top 3"
        print(f"{row['type']:13} best score {pack['best_score']:.2f}   {mark:28} {row['question']}")

Compare the **best scores** with those of real questions (usually higher, but not always). A score cut-off alone
would also throw away real questions — that's why the AI always gets the capability cards and decides itself.

### D — Telugu and Hinglish

The same business questions, in Telugu and Hinglish. Do they still get what they need?

In [ ]:
golden_by_id = {row["qid"]: row for row in golden_questions}

for row in language_questions:
    must_have = split_ids(golden_by_id[row["qid"]]["must_find"])
    pack = context_pack(row["question"])
    in_pack = pack["found_by_search"] + pack["followed_links"] + pack["always_added"]
    print(f"{row['qid']}  {row['language']:9} completeness {share_in(must_have, in_pack):.2f}   {row['question']}")

### E — Schema questions: easy vs real wording

`schema_questions` come from the workbook. Most of them contain the column's own name ("How many orders per
payment method?") — easy for any search. Only the real-wording ones are a fair test.

In [ ]:
easy = []
real = []
for row in schema_questions:
    if row["names_the_record"] == "Y":
        easy.append(row)
    else:
        real.append(row)

for name, questions in (("names the column", easy), ("real wording", real)):
    results = check_all("schema", questions, SCHEMA_TABS)
    print(f"{name:17} in top 3: {count_passed(results) / len(results):.2f}   ({len(results)} questions)")

### F — What changed since the last run?

After you fix something and run again, this shows which questions changed — so you also catch a fix that broke
something else.

In [ ]:
this_run = {}
for results in (golden_results, glossary_results, example_results, join_results):
    for result in results:
        this_run[result["set"] + " " + str(result["qid"])] = result

# Save this run, then compare with the one before it.
run_file = f"eval_runs/{run_name}.csv"
with open(run_file, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["key", "question", "passed"])
    for key, result in this_run.items():
        writer.writerow([key, result["question"], result["passed"]])

earlier = sorted(name for name in os.listdir("eval_runs") if name.startswith("run_") and name != run_name + ".csv")
if not earlier:
    print("This is the first run — fix something, run again, then come back to this cell.")
else:
    with open("eval_runs/" + earlier[-1], encoding="utf-8") as file:
        before = {row["key"]: row["passed"] for row in csv.DictReader(file)}
    for key, result in this_run.items():
        if key in before and before[key] != str(result["passed"]):
            change = "✅ now passes" if result["passed"] == 1 else "❌ now fails"
            print(f"{change}   {key}   {result['question']}")
    print("Compared with", earlier[-1])